# ForestWatch Papua — Improve Model 5-Kelas (skema final, gabungan kelas)

Gabungkan **Tambang → Lahan Terbuka** (kemiripan spektral tanah/batuan terbuka, Tang & Werner
2023 — tambang permukaan ber-NDVI rendah serupa lahan terbuka) dan **Sawit + Pertanian Lain →
Pertanian** (sama-sama kanopi pertanian). Hasil: **5 kelas** — Perairan, Hutan, Lahan Terbuka,
Pertanian, Permukiman. Ini skema final proyek (bukan eksperimen sampingan lagi).

## Cara kerja
Notebook ini me-remap label 7-kelas → 5-kelas **on-the-fly saat training** (lihat
`REMAP_7_TO_5` di `forestwatch.constants`), TANPA re-export GEE atau re-cut patch. Dataset
Kaggle yang sama (`fw-papua-train-v4-*`, `fw-papua-val-v4`, `fw-papua-test-v4`) dipakai apa
adanya. Model **warm-start dari checkpoint 7-kelas** (`model_1_attention_unet`): encoder+
decoder di-load, HANYA `segmentation_head` (beda ukuran output 7 vs 5) diinisialisasi baru.

## Catatan jujur (penting, utk esai)
Menggabungkan kelas SECARA MEKANIS menaikkan mIoU (lebih sedikit kelas sulit, rata-rata lebih
mudah) — dari confusion matrix fine-tune 7-kelas yang sudah ada, terbukti:
- **Tambang↔Lahan Terbuka**: ~75% piksel Tambang yang salah ternyata memang ketuker ke Lahan
  Terbuka — penggabungan ini **memperbaiki kebingungan nyata**, bukan cuma menyederhanakan soal.
- **Sawit↔Pertanian Lain**: cuma ~2% piksel Sawit yang ketuker ke Pertanian Lain — penggabungan
  ini lebih ke **pengelompokan tematik** (sama-sama "pertanian") daripada perbaikan kebingungan
  nyata; kenaikan mIoU di sini lebih dominan efek "rata-rata 5 kelas lebih mudah dari 7".
- **Hutan->Tambang dan Hutan->Lahan Terbuka jadi SATU transisi** (`hutan_ke_lahan_terbuka`),
  begitu juga **Hutan->Sawit dan Hutan->Pertanian Lain jadi SATU** (`hutan_ke_pertanian`) —
  analisis driver deforestasi spesifik-komoditas hilang di skema ini. Trade-off ini harus
  disebut jujur kalau hasil 5-kelas ini dipakai di esai/WebGIS.

## Metode training
Identik dgn `improve_model.ipynb` v2 (full fine-tune encoder dibuka, LR bertingkat Howard &
Ruder 2018, loss median-frequency weighted Eigen & Fergus 2015 + Focal-Tversky, sampler
frequency-proportional, guard checkpoint, 40 epoch) **+ logit adjustment** (Menon dkk., *"Long-
tail Learning via Logit Adjustment"*, ICLR 2021) — target khusus utk masalah yang terbukti dari
confusion matrix 7-kelas: piksel kelas lemah sering "ketuker ke Hutan" (kelas mayoritas, ~43%
piksel). Logit adjustment mengoreksi bias keputusan ini langsung saat training (logit ditambah
`tau * log_prior`, negatif utk kelas jarang, sblm loss -- memaksa model belajar margin lebih
besar khusus kelas jarang), bukan cuma menambah bobot gradien spt median-frequency weighting
saja. Lihat cell `dist5cls`/`build5`/`finetune5` utk detail implementasi.


## Bagian 0 — Setup environment (Colab / Lab / Kaggle)

In [ ]:
# === Bagian 0 -- Setup (set ENV = "colab" / "lab" / "kaggle") ===
ENV = "colab"   # "colab" | "lab" (PC+Drive Desktop) | "kaggle"
import os, sys, subprocess, importlib
from pathlib import Path

# Auto-deteksi Kaggle (folder /kaggle hanya ada di runtime Kaggle).
if Path("/kaggle").exists() and ENV != "kaggle":
    print(f"[auto-detect] /kaggle -> override ENV='{ENV}' -> 'kaggle'")
    ENV = "kaggle"

DRIVE_ROOT = None
_SRC = None
if ENV == "colab":
    subprocess.run("cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q -e /content/fw_repo[ml]", shell=True, check=False)
    _SRC = "/content/fw_repo/model/src"
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN mount Drive Desktop lab
elif ENV == "kaggle":
    subprocess.run("cd /kaggle/working && (git -C fw_repo pull -q || git clone --depth 1 "
                   "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
                   shell=True, check=False)
    subprocess.run("pip install -q --no-deps -e /kaggle/working/fw_repo[ml]", shell=True, check=False)
    subprocess.run("pip install -q --no-deps segmentation-models-pytorch albumentations torchmetrics",
                   shell=True, check=False)
    _SRC = "/kaggle/working/fw_repo/model/src"
else:
    raise ValueError("ENV harus 'colab' | 'lab' | 'kaggle'")

if _SRC and _SRC not in sys.path:
    sys.path.insert(0, _SRC)
for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | CUDA={torch.cuda.is_available()}{_gpu}")


In [ ]:
# === Deklarasi path, identitas model, & config (eksperimen 5-kelas) ===
from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
from forestwatch.constants import (
    N_CLASSES_5, CLASS_NAMES_5, CLASS_COLORS_5, REMAP_7_TO_5,
)
cfg = load_config()

MODEL_KEY    = "model_1_attention_unet"          # baseline 7-kelas dipakai sbg warm-start
MODEL_ARCH   = dict(architecture="unet_scse", encoder_name="resnet50")
EXPERIMENT   = "model_1_attention_unet_5class"   # nama folder output, terpisah dari v2 7-kelas
STRONG = [0, 1]      # Perairan, Hutan -- JAGA (jangan turun)
WEAK   = [2, 3, 4]   # Lahan Terbuka(gabung Tambang), Pertanian(gabung Sawit), Permukiman

DRIVE_TARGET_NAME = f"ForestWatch_Outputs/Model_Improve_5class/{MODEL_KEY}"

if ENV in ("colab", "lab"):
    BAHAN_DIR   = DRIVE_ROOT / "Bahan_Training_Fix_Combined_v4"
    MODELS_ROOT = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Comparison"
    BASE_DIR    = MODELS_ROOT / MODEL_KEY
    BASE_CKPT   = BASE_DIR / "best_model.pt"
    FT_DIR      = DRIVE_ROOT / "ForestWatch_Outputs" / "Model_Improve_5class" / MODEL_KEY
else:  # kaggle: data + checkpoint baseline dari dataset yg di-attach (Add Input)
    BAHAN_DIR = None                       # diisi di cell ekstrak (rglob)
    BASE_DIR  = None
    BASE_CKPT = next(Path("/kaggle/input").rglob("best_model.pt"))
    FT_DIR    = Path("/kaggle/working") / "Model_Improve_5class" / MODEL_KEY

FT_DIR.mkdir(parents=True, exist_ok=True)
FT_CKPT    = FT_DIR / "best_model_finetune_5class.pt"
FT_RESUME  = FT_DIR / "best_model_finetune_5class_resume.pt"
FT_SAMPLER = FT_DIR / "patch_sampler_5class.json"
OUT_DIR    = FT_DIR

assert BASE_CKPT.exists(), f"Baseline checkpoint (7-kelas) tak ada: {BASE_CKPT}"
print("Baseline ckpt (7-kelas, warm-start) :", BASE_CKPT)
print("Dataset                              :", BAHAN_DIR if BAHAN_DIR else "(kaggle, attach Bahan_Training_Fix_Combined_v4)")
print("Output FT dir                        :", FT_DIR)
print("Kelas 5 (baru)                       :", CLASS_NAMES_5)
print("Remap 7->5                           :", REMAP_7_TO_5)
print("Kelas JAGA   :", [CLASS_NAMES_5[c] for c in STRONG])
print("Kelas target :", [CLASS_NAMES_5[c] for c in WEAK])


In [ ]:
# === Ekstrak dataset ke disk lokal + daftar file (train/val/test) ===
# IDENTIK dgn improve_model.ipynb v2 -- data .npz mentah TETAP 7-kelas, remap ke 5-kelas
# dilakukan di cell-cell berikutnya (on-the-fly), bukan di sini.
from forestwatch.data.dataset import extract_dataset_archives
from forestwatch.data.patches import list_patches

if ENV in ("colab", "lab"):
    LOCAL_DIR = Path("/content/dataset_local") if ENV == "colab" else Path.home() / "dataset_local"
    _splits = ("train", "val", "test")
    if (BAHAN_DIR / "train_rajaampat").exists():
        _splits = _splits + ("train_rajaampat",)
    local_dirs = extract_dataset_archives(BAHAN_DIR, LOCAL_DIR, splits=_splits, max_workers=8)
else:  # kaggle
    BAHAN_SRC = Path("/kaggle/temp/bahan_src"); BAHAN_SRC.mkdir(parents=True, exist_ok=True)
    for s in ("train", "val", "test", "train_rajaampat"):
        (BAHAN_SRC / s).mkdir(parents=True, exist_ok=True)

    for npz in Path("/kaggle/input").rglob("*.npz"):
        parts_lower = [p.lower() for p in npz.parts]
        path_str = str(npz).lower()
        if any(p.startswith("train_part") for p in parts_lower):
            split = "train"
        elif "rajaampat" in path_str:
            split = "train_rajaampat"
        elif "fw-papua-val" in path_str or any(p == "val" for p in parts_lower):
            split = "val"
        elif "fw-papua-test" in path_str or any(p == "test" for p in parts_lower):
            split = "test"
        else:
            continue
        _dst = BAHAN_SRC / split / npz.name
        if not _dst.exists():
            os.symlink(npz, _dst)

    for s in ("train", "val", "test", "train_rajaampat"):
        n = len(list((BAHAN_SRC / s).glob("*.npz")))
        if n:
            print(f"{s}: {n} file .npz ditemukan -> {BAHAN_SRC / s}")

    _splits = tuple(s for s in ("train", "val", "test", "train_rajaampat")
                     if next((BAHAN_SRC / s).glob("*.npz"), None) is not None)
    local_dirs = {s: BAHAN_SRC / s for s in _splits}

final_train_files = list_patches(local_dirs["train"])
if "train_rajaampat" in local_dirs:
    _ra = list_patches(local_dirs["train_rajaampat"])
    final_train_files = final_train_files + _ra
    print(f"  + {len(_ra)} patch Raja Ampat (Tambang asli)")
val_p  = list_patches(local_dirs["val"])
test_p = list_patches(local_dirs["test"])
assert final_train_files and val_p and test_p, "train/val/test kosong -- cek BAHAN_DIR / attach dataset."
print(f"train={len(final_train_files)} | val={len(val_p)} | test={len(test_p)}")


In [ ]:
# === Distribusi 5-kelas (gabungan dari hitung 7-kelas) + bobot median-frequency + log-prior ===
# compute_class_distribution menghitung label MENTAH (0..6, 7-kelas) dari .npz -- dipertahankan
# apa adanya (tidak remap dulu, lebih cepat: 1x scan dipakai utk 2 keperluan). Hasil count
# DIGABUNG jadi 5-kelas via REMAP_7_TO_5, baru median_frequency_weights dihitung di ruang 5-kelas.
import json
import numpy as np
from forestwatch.data.patches import compute_class_distribution
from forestwatch.training.metrics import median_frequency_weights

_dist7_cache = FT_DIR / "train_distribution_7raw.json"
if _dist7_cache.exists():
    train_dist_7 = {int(k): int(v) for k, v in json.load(open(_dist7_cache)).items()}
    print("Distribusi train (7-kelas mentah) dimuat dari cache:", _dist7_cache.name)
else:
    train_dist_7 = compute_class_distribution(local_dirs["train"], n_classes=7)
    save_json({str(k): int(v) for k, v in train_dist_7.items()}, _dist7_cache)

train_dist_5 = {c: 0 for c in range(N_CLASSES_5)}
for old_c, px in train_dist_7.items():
    train_dist_5[REMAP_7_TO_5[old_c]] += px

class_w_5 = median_frequency_weights(train_dist_5, n_classes=N_CLASSES_5)
# Ekspansi balik ke panjang-7 (utk compute_patch_sampler_weights yg baca label MENTAH 0..6) --
# old_id dapat bobot dari kelas-5 hasil gabungannya, bukan dihitung ulang per-old-id.
class_w_7_expanded = [class_w_5[REMAP_7_TO_5[c]] for c in range(7)]

# Log-prior per kelas (5-kelas) -- dipakai cell `build5`/`finetune5` utk LOGIT ADJUSTMENT
# (Menon dkk., "Long-tail Learning via Logit Adjustment", ICLR 2021). Bukan pengganti loss
# berbobot di atas, tapi pelengkap: loss berbobot menekan GRADIEN kelas jarang lebih besar,
# logit adjustment langsung mengoreksi BIAS KEPUTUSAN model thd kelas mayoritas (Hutan) saat
# training -- target masalah "weak class ketuker ke Hutan" yg terbukti dari confusion matrix
# fine-tune 7-kelas (mis. 18-19% Lahan Terbuka & Pertanian Lain ketuker ke Hutan).
log_prior_5 = np.log(np.array([train_dist_5[c] for c in range(N_CLASSES_5)], dtype=np.float64))
log_prior_5 = log_prior_5 - log_prior_5.max()  # stabilitas numerik (offset tak pengaruh ranking)

_tot5 = sum(train_dist_5.values()) or 1
print(f"{'kelas (5)':<16}{'piksel':>16}{'freq%':>9}{'bobot':>9}{'log_prior':>11}")
for c in range(N_CLASSES_5):
    n = train_dist_5.get(c, 0)
    print(f"{CLASS_NAMES_5[c]:<16}{n:>16,}{100*n/_tot5:>8.2f}%{class_w_5[c]:>9.3f}{log_prior_5[c]:>11.3f}")
print("\nclass_w_5 (median-frequency) ->", [round(w, 3) for w in class_w_5])
print("class_w_7_expanded (utk sampler, panjang 7) ->", [round(w, 3) for w in class_w_7_expanded])
print("log_prior_5 (utk logit adjustment) ->", [round(float(w), 3) for w in log_prior_5])


In [ ]:
import numpy as np
# === Sampler: frequency-proportional, dihitung dari label 7-kelas MENTAH + bobot 5-kelas ===
# compute_patch_sampler_weights membaca label .npz APA ADANYA (0..6) -- pakai class_w_7_expanded
# (bobot 5-kelas yg sudah "dipinjam" ke index lama) supaya hasilnya konsisten dgn skema 5-kelas,
# tanpa perlu mengubah fungsi generik di package atau remap file .npz.
from forestwatch.data.dataset import compute_patch_sampler_weights

train_sampler_weights = compute_patch_sampler_weights(
    final_train_files, class_w_7_expanded, n_classes=7, cache_path=FT_SAMPLER,
)
_w = np.array(train_sampler_weights)
print(f"Bobot sampler: n={len(_w):,} | min={_w.min():.3e} | median={np.median(_w):.3e} | max={_w.max():.3e}")


In [ ]:
# === LOGIT ADJUSTMENT (Menon dkk., ICLR 2021) -- lihat penjelasan di cell `dist5cls` ===
# Dipakai HANYA saat menghitung loss training (cell `finetune5`): logit ditambah
# tau*log_prior (NEGATIF utk kelas jarang) sblm masuk loss -- ini membuat loss "lebih ketat"
# utk kelas jarang (perlu logit mentah lebih besar utk loss kecil), memaksa model belajar
# margin lebih besar khusus kelas jarang. Saat eval/val/test, logit MENTAH (tanpa adjustment)
# dipakai utk argmax -- koreksinya sudah "terbakar" ke bobot model lewat training, sesuai
# resep paper (varian train-time logit-adjusted loss, BUKAN varian post-hoc-saat-inferensi).
LOGIT_ADJ_TAU = 1.0  # naikkan (mis. 1.5) kalau weak class masih kalah; turunkan (mis. 0.5)
                      # kalau guard sering gagal (mayoritas terlalu tertekan)
LOG_PRIOR_5_T = torch.tensor(log_prior_5, dtype=torch.float32, device=device).view(1, -1, 1, 1)

def apply_logit_adjustment(logits):
    return logits + LOGIT_ADJ_TAU * LOG_PRIOR_5_T

print(f"Logit adjustment AKTIF (tau={LOGIT_ADJ_TAU}) -- hanya di loss training, bukan di eval.")


## Baseline (BEFORE, 5-kelas) — tetapkan floor kelas mayoritas (anti-bocor: VAL utk guard, TEST utk laporan)

In [ ]:
# === Baseline (BEFORE) per-kelas IoU 5-KELAS: VAL (utk guard) + TEST (utk laporan akhir) ===
# Label ground-truth (d["lab"]) MASIH 7-kelas mentah dari .npz -- di-remap ke 5-kelas via
# REMAP_LUT_NP sblm dibandingkan dgn prediksi model (yg outputnya sudah 5-kelas).
import numpy as np, torch, gc
from forestwatch.training.metrics import compute_confusion_matrix, metric_summary

def _eval_files(mdl, files):
    mdl.eval()
    cm = np.zeros((N_CLASSES_5, N_CLASSES_5), dtype=np.int64)
    with torch.no_grad():
        for i, fp in enumerate(files):
            d = np.load(fp)
            img = torch.from_numpy(d["img"]).float().unsqueeze(0).to(device)
            pr = mdl(img).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
            lab5 = REMAP_LUT_NP[np.asarray(d["lab"]).astype(np.int64)]
            cm += compute_confusion_matrix(pr, lab5, n_classes=N_CLASSES_5)
            del d, img, pr
            if i % 3000 == 0:
                gc.collect(); torch.cuda.empty_cache()
    return metric_summary(cm, class_names=CLASS_NAMES_5)

base_val  = _eval_files(model, val_p)
base_test = _eval_files(model, test_p)
base_val_iou  = [r["iou"] for r in base_val["per_class"]]
base_test_iou = [r["iou"] for r in base_test["per_class"]]

EPS = 0.01
floor = {c: base_val_iou[c] - EPS for c in STRONG}
val_miou = base_val["mean_iou"]
test_miou = base_test["mean_iou"]
test_fwiou = base_test["fwiou"]
print(f"Baseline (5-kelas) VAL mIoU={val_miou:.4f} | TEST mIoU={test_miou:.4f} | TEST FWIoU={test_fwiou:.4f}")
print("Floor mayoritas (VAL):", {CLASS_NAMES_5[c]: round(floor[c], 4) for c in STRONG})
header_kelas, header_val, header_test = "kelas", "VAL IoU", "TEST IoU"
print(f"\n{header_kelas:<16}{header_val:>9}{header_test:>10}")
for c in range(N_CLASSES_5):
    print(f"{CLASS_NAMES_5[c]:<16}{base_val_iou[c]:>9.4f}{base_test_iou[c]:>10.4f}")


## Fine-tune 5-kelas (encoder DIBUKA, LR bertingkat) — seleksi checkpoint terjaga + resume-safe

In [ ]:
# === Fine-tune 5-KELAS (encoder dibuka) -- seleksi checkpoint TERJAGA (guarded) ===
# Identik metodenya dgn improve_model.ipynb v2; beda: (1) target y di-remap 7->5 tiap batch via
# REMAP_LUT sblm loss/metric (karena PapuaDataset/label .npz masih mentah 7-kelas), (2) loss
# TRAINING dihitung dari logit yg sudah di-logit-adjustment (apply_logit_adjustment, cell
# `build5`) -- val/test TETAP pakai logit mentah, sesuai resep Menon dkk. 2021.
import time, torch
from torch.amp import autocast
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.auto import tqdm
from forestwatch.training.metrics import set_seed
try:
    from torch.amp import GradScaler; _NEWSCALER = True
except ImportError:
    from torch.cuda.amp import GradScaler; _NEWSCALER = False

FT_LR_ENC, FT_LR_HEAD = 1e-5, 1e-4; FT_EPOCHS = 40; FT_PATIENCE = 12
set_seed(cfg["project"]["seed"])
use_amp = torch.cuda.is_available()

enc_params  = [p for n, p in model.named_parameters()
               if n.startswith("encoder.") and p.requires_grad]
head_params = [p for n, p in model.named_parameters()
               if not n.startswith("encoder.") and p.requires_grad]
assert enc_params and head_params, "param-group kosong (encoder/head) -- cek nama parameter model"
trainable = enc_params + head_params
optimizer = torch.optim.AdamW(
    [{"params": enc_params, "lr": FT_LR_ENC},
     {"params": head_params, "lr": FT_LR_HEAD}],
    weight_decay=cfg["training"]["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FT_EPOCHS)
print(f"Optimizer: AdamW 2 grup -- encoder lr={FT_LR_ENC:.0e} ({len(enc_params)} tensor), "
      f"head lr={FT_LR_HEAD:.0e} ({len(head_params)} tensor) | {FT_EPOCHS} epoch, patience {FT_PATIENCE}")
scaler = (GradScaler(device.type) if _NEWSCALER else GradScaler()) if use_amp else None
iou_pc = MulticlassJaccardIndex(num_classes=N_CLASSES_5, average=None).to(device)

start_epoch, best_obj, best_epoch, wait, history = 1, -1.0, -1, 0, []
if FT_RESUME.exists():
    try:
        st = torch.load(FT_RESUME, map_location=device)
        model.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"])
        if scaler and st.get("scaler"): scaler.load_state_dict(st["scaler"])
        start_epoch = st["epoch"] + 1; best_obj = st["best_obj"]; best_epoch = st["best_epoch"]
        wait = st["wait"]; history = st["history"]
        set_encoder_trainable(model, True)
        print(f"Resume -> epoch {start_epoch} (best guarded val mIoU={best_obj:.4f})")
    except Exception as e:
        print("Gagal resume:", e)

for ep in range(start_epoch, FT_EPOCHS + 1):
    model.train()
    tr = 0.0; t0 = time.time()
    for x, y in tqdm(train_loader, desc=f"ep{ep:02d} train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        y = REMAP_LUT[y.long()]
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with autocast(device_type=device.type):
                loss = loss_fn(apply_logit_adjustment(model(x)), y)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss = loss_fn(apply_logit_adjustment(model(x)), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0); optimizer.step()
        tr += float(loss.item())
    scheduler.step(); tr /= max(len(train_loader), 1)

    model.eval(); iou_pc.reset(); vl = 0.0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"ep{ep:02d} val", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y = REMAP_LUT[y.long()]
            p = model(x); vl += float(loss_fn(p, y).item()); iou_pc.update(p.argmax(1), y)
    vl /= max(len(val_loader), 1)
    pc = [float(v) for v in iou_pc.compute().tolist()]
    vmiou = sum(pc) / len(pc); weak_miou = sum(pc[c] for c in WEAK) / len(WEAK)
    strong_ok = all(pc[c] >= floor[c] for c in STRONG)
    history.append({"epoch": ep, "train_loss": tr, "val_loss": vl, "val_miou": vmiou,
                    "weak_miou": weak_miou, "val_iou_per_class": [round(v, 4) for v in pc],
                    "strong_ok": bool(strong_ok), "lr": optimizer.param_groups[0]["lr"],
                    "epoch_time_sec": time.time() - t0})
    print(f"ep{ep:02d} | loss {tr:.4f} | val {vl:.4f} | mIoU {vmiou:.4f} | "
          f"weak {weak_miou:.4f} | mayoritas_ok={strong_ok}")
    print("   IoU/kelas:", [round(v, 3) for v in pc])

    if strong_ok and vmiou > best_obj:
        best_obj, best_epoch, wait = vmiou, ep, 0
        torch.save(model.state_dict(), FT_CKPT)
        print(f"   -> best guarded checkpoint (val mIoU={vmiou:.4f}) -> {FT_CKPT.name}")
    else:
        wait += 1
    torch.save({"epoch": ep, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict() if scaler else None,
                "best_obj": best_obj, "best_epoch": best_epoch, "wait": wait,
                "history": history}, FT_RESUME)
    if wait >= FT_PATIENCE:
        print(f"Early stopping di epoch {ep} (patience={FT_PATIENCE})."); break

print(f"\nSelesai. best guarded val mIoU={best_obj:.4f} @ epoch {best_epoch}")
if best_epoch < 0:
    print("PERINGATAN: TAK ada epoch yg lolos guard (semua menurunkan kelas mayoritas).")
    print("-> FT_CKPT tidak tersimpan. Pertahankan baseline; coba turunkan FT_LR_ENC atau LOGIT_ADJ_TAU. JANGAN promosikan.")


In [ ]:
# === Plot kurva fine-tune 5-kelas -> FT_DIR ===
import matplotlib.pyplot as plt
assert history, "history kosong -- jalankan cell fine-tune dulu."
eps = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].plot(eps, [h["train_loss"] for h in history], label="train", lw=2)
ax[0].plot(eps, [h["val_loss"] for h in history], label="val", lw=2)
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(eps, [h["val_miou"] for h in history], color="green", lw=2, label="val mIoU")
ax[1].plot(eps, [h["weak_miou"] for h in history], color="red", lw=2, label="weak mIoU")
ax[1].axhline(base_val["mean_iou"], color="gray", ls="--", label="baseline mIoU")
ax[1].set_title("mIoU (val, 5-kelas)"); ax[1].set_ylim(0, 1); ax[1].legend(); ax[1].grid(alpha=0.3)
for c in range(N_CLASSES_5):
    ax[2].plot(eps, [h["val_iou_per_class"][c] for h in history],
               color=CLASS_COLORS_5[c], lw=1.6, label=CLASS_NAMES_5[c])
ax[2].set_title("val IoU per-kelas (5-kelas)"); ax[2].set_ylim(0, 1)
ax[2].legend(fontsize=7, ncol=2); ax[2].grid(alpha=0.3)
fig.suptitle(EXPERIMENT + " (fine-tune 5-kelas)"); fig.tight_layout()
fig.savefig(FT_DIR / "training_curve_finetune_5class.png", dpi=120, bbox_inches="tight"); plt.show()
print("Disimpan:", FT_DIR / "training_curve_finetune_5class.png")


## Evaluasi akhir di TEST 5-kelas (sekali) — tabel BEFORE/AFTER + vonis jujur

In [ ]:
# === Evaluasi TEST akhir 5-KELAS (sekali) + tabel BEFORE/AFTER + vonis jujur ===
import numpy as np, torch
import matplotlib.pyplot as plt
from forestwatch.model.architecture import build_unet, export_to_onnx

if not FT_CKPT.exists():
    print("FT_CKPT tak ada -> fine-tune tak menghasilkan model yg lolos guard.")
    print("VONIS: pertahankan BASELINE warm-start. Tidak ada artefak fine-tune utk dipromosikan.")
else:
    ft_model = build_unet(in_channels=cfg["model"]["in_channels"], classes=N_CLASSES_5,
                          encoder_weights=None, **MODEL_ARCH).to(device)
    ft_model.load_state_dict(torch.load(FT_CKPT, map_location="cpu"))
    ft_test = _eval_files(ft_model, test_p)
    ft_iou = [r["iou"] for r in ft_test["per_class"]]

    header_kelas, header_before, header_after, header_delta = "kelas", "BEFORE", "AFTER", "delta"
    print(f"{header_kelas:<16}{header_before:>9}{header_after:>9}{header_delta:>9}")
    drop_majority = []
    for c in range(N_CLASSES_5):
        dlt = ft_iou[c] - base_test_iou[c]
        flag = ""
        if c in STRONG and dlt < -EPS:
            flag = "  <- MAYORITAS TURUN"; drop_majority.append(c)
        elif c in WEAK and dlt > 0:
            flag = "  <- naik"
        print(f"{CLASS_NAMES_5[c]:<16}{base_test_iou[c]:>9.4f}{ft_iou[c]:>9.4f}{dlt:>+9.4f}{flag}")

    base_miou, ft_miou = base_test["mean_iou"], ft_test["mean_iou"]
    base_fwiou, ft_fwiou = base_test["fwiou"], ft_test["fwiou"]
    print(f"\nmIoU  : {base_miou:.4f} -> {ft_miou:.4f} ({ft_miou - base_miou:+.4f})")
    print(f"FWIoU : {base_fwiou:.4f} -> {ft_fwiou:.4f} ({ft_fwiou - base_fwiou:+.4f})")

    mi_up = ft_test["mean_iou"] >= base_test["mean_iou"]
    if mi_up and not drop_majority:
        print("\nVONIS: BERHASIL -- mIoU naik & kelas mayoritas tak turun.")
    else:
        why = []
        if not mi_up: why.append("mIoU TIDAK naik di test")
        if drop_majority: why.append("mayoritas turun: " + ", ".join(CLASS_NAMES_5[c] for c in drop_majority))
        print("\nVONIS: BELUM memenuhi target (" + "; ".join(why) + "). JANGAN promosikan.")

    save_json(ft_test, FT_DIR / "metrics_finetune_5class.json")
    save_json({"model_key": EXPERIMENT, **MODEL_ARCH,
               "method": "5-kelas gabungan (Tambang->Lahan Terbuka, Sawit+Pertanian Lain->Pertanian) "
                         "+ warm-start dari checkpoint 7-kelas (encoder+decoder) + full fine-tune "
                         "LR bertingkat + median-frequency weighted focal+tversky+CE + sampler "
                         "frequency-proportional + logit adjustment (Menon dkk. ICLR 2021, "
                         f"tau={LOGIT_ADJ_TAU})",
               "class_names": list(CLASS_NAMES_5), "remap_7_to_5": REMAP_7_TO_5,
               "dataset": BAHAN_DIR.name if BAHAN_DIR else "kaggle_attached_v4",
               "ft_lr_encoder": FT_LR_ENC, "ft_lr_head": FT_LR_HEAD,
               "encoder_frozen": False, "loss_class_weighted": True,
               "logit_adjustment_tau": LOGIT_ADJ_TAU,
               "class_weights_median_freq_5": [round(float(w), 4) for w in class_w_5],
               "best_epoch_guarded": best_epoch,
               "baseline_test_miou": base_test["mean_iou"], "finetune_test_miou": ft_test["mean_iou"],
               "baseline_test_per_class_iou": base_test_iou, "finetune_test_per_class_iou": ft_iou,
               "per_class": ft_test["per_class"]}, FT_DIR / "summary_finetune_5class.json")

    cm = np.array(ft_test["confusion_matrix"]); cmn = cm / cm.sum(axis=1, keepdims=True).clip(1)
    fig, axx = plt.subplots(figsize=(7, 5.5)); im = axx.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(N_CLASSES_5):
        for j in range(N_CLASSES_5):
            axx.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=9,
                     color="white" if cmn[i, j] > 0.5 else "black")
    axx.set_xticks(range(N_CLASSES_5)); axx.set_xticklabels(CLASS_NAMES_5, rotation=45, ha="right")
    axx.set_yticks(range(N_CLASSES_5)); axx.set_yticklabels(CLASS_NAMES_5)
    axx.set_xlabel("Predicted"); axx.set_ylabel("True"); axx.set_title("Confusion - fine-tune 5-kelas")
    fig.colorbar(im, ax=axx); fig.tight_layout()
    fig.savefig(FT_DIR / "confusion_matrix_finetune_5class.png", dpi=120, bbox_inches="tight"); plt.show()

    try:
        export_to_onnx(ft_model, FT_DIR / "model_finetune_5class.onnx",
                       in_channels=cfg["model"]["in_channels"], patch_size=cfg["inference"]["patch_size"])
        print("ONNX:", FT_DIR / "model_finetune_5class.onnx")
    except Exception as e:
        print("ONNX dilewati:", e)
    print("Disimpan:", FT_DIR / "metrics_finetune_5class.json",
          "| summary_finetune_5class.json | confusion_matrix_finetune_5class.png")


In [ ]:
# === Paket hasil utk diunduh (Kaggle tak ada mount Drive) ===
import shutil

if ENV not in ("colab", "lab"):
    zip_base = FT_DIR.parent / (FT_DIR.name + "_5class_package")
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=FT_DIR)
    print("Paket siap diunduh:", zip_path)
    print("Langkah selanjutnya (manual, Kaggle tak ada mount Drive):")
    print("  1. Kaggle -> Save Version -> tunggu selesai -> buka tab 'Output' notebook ini.")
    print(f"  2. Unduh '{Path(zip_path).name}', lalu di Google Drive buat folder:")
    print(f"     Satria Data 3.0/{DRIVE_TARGET_NAME}")
    print("  3. Ekstrak isi zip ke folder itu.")
else:
    print("Hasil fine-tune 5-kelas tersimpan lengkap di:", FT_DIR)
    print("(Eksperimen terpisah -- TIDAK dipromosikan otomatis ke baseline manapun.)")
